# 04 — Economic Activity
**Source:** India Districts Census 2011 · 640 districts  
Covers worker categories, dependency ratios, and household income (power parity) distribution.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.workers, f.male_workers, f.female_workers,
           f.main_workers, f.marginal_workers, f.non_workers,
           f.cultivator_workers, f.agricultural_workers,
           f.household_workers, f.other_workers,
           f.power_parity_less_than_rs_45000,
           f.power_parity_rs_45000_90000,
           f.power_parity_rs_90000_150000,
           f.power_parity_rs_150000_240000,
           f.power_parity_rs_240000_330000,
           f.power_parity_rs_330000_425000,
           f.power_parity_rs_425000_545000,
           f.power_parity_above_rs_545000,
           f.total_power_parity
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df = df[df['population'] > 0]
df['worker_participation'] = (df['workers'] / df['population'] * 100).round(2)
df['female_wp_rate']       = (df['female_workers'] / df['population'] * 100).round(2)
df['male_wp_rate']         = (df['male_workers'] / df['population'] * 100).round(2)
df['dependency_ratio']     = (df['non_workers'] / df['workers'].replace(0, np.nan) * 100).round(2)

state = df.groupby('state_name').sum(numeric_only=True).reset_index()
state['worker_participation'] = (state['workers'] / state['population'] * 100).round(2)
state['female_wp_rate']       = (state['female_workers'] / state['population'] * 100).round(2)
state['male_wp_rate']         = (state['male_workers'] / state['population'] * 100).round(2)
state['cultivator_pct']       = (state['cultivator_workers'] / state['workers'] * 100).round(2)
state['agri_pct']             = (state['agricultural_workers'] / state['workers'] * 100).round(2)
state['household_pct']        = (state['household_workers'] / state['workers'] * 100).round(2)
state['other_pct']            = (state['other_workers'] / state['workers'] * 100).round(2)

print(f'National Worker Participation Rate: {df.workers.sum()/df.population.sum()*100:.2f}%')

## 4.1 — Male vs Female Worker Participation Rate by State

In [ ]:
state_sorted = state.sort_values('female_wp_rate')

fig = go.Figure()
fig.add_trace(go.Bar(name='Male WP Rate',   y=state_sorted['state_name'], x=state_sorted['male_wp_rate'],   orientation='h', marker_color='#3498db'))
fig.add_trace(go.Bar(name='Female WP Rate', y=state_sorted['state_name'], x=state_sorted['female_wp_rate'], orientation='h', marker_color='#e91e63'))

fig.update_layout(
    barmode='group',
    title='Male vs Female Worker Participation Rate by State',
    xaxis_title='Workers as % of Total Population',
    height=700,
    legend=dict(orientation='h', y=1.02)
)
fig.show()

## 4.2 — Worker Type Breakdown by State (Stacked %)

In [ ]:
state_sorted2 = state.sort_values('cultivator_pct', ascending=False)

fig = go.Figure()
colors_worker = ['#27ae60','#f39c12','#e74c3c','#3498db']
for col, label, color in zip(['cultivator_pct','agri_pct','household_pct','other_pct'],
                              ['Cultivators','Agricultural Labourers','Household Industry','Other Workers'],
                              colors_worker):
    fig.add_trace(go.Bar(name=label, x=state_sorted2['state_name'], y=state_sorted2[col], marker_color=color))

fig.update_layout(
    barmode='stack',
    title='Worker Type Distribution by State (%) — sorted by Cultivator Share',
    yaxis_title='% of Workers',
    xaxis_tickangle=-45,
    height=530,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 4.3 — Worker Type National Pie Chart

In [ ]:
worker_nat = {
    'Cultivators':       df['cultivator_workers'].sum(),
    'Agri Labourers':    df['agricultural_workers'].sum(),
    'Household Industry':df['household_workers'].sum(),
    'Other Workers':     df['other_workers'].sum(),
}

fig = px.pie(
    names=list(worker_nat.keys()),
    values=list(worker_nat.values()),
    title='National Worker Composition — 2011 Census',
    color_discrete_sequence=['#27ae60','#f39c12','#e74c3c','#3498db'],
    hole=0.35
)
fig.update_traces(textinfo='percent+label')
fig.update_layout(height=450)
fig.show()

## 4.4 — Power Parity (Income) Distribution — National Stacked Bar

In [ ]:
pp_cols = [
    ('power_parity_less_than_rs_45000',    '<₹45K'),
    ('power_parity_rs_45000_90000',        '₹45K–90K'),
    ('power_parity_rs_90000_150000',       '₹90K–1.5L'),
    ('power_parity_rs_150000_240000',      '₹1.5L–2.4L'),
    ('power_parity_rs_240000_330000',      '₹2.4L–3.3L'),
    ('power_parity_rs_330000_425000',      '₹3.3L–4.25L'),
    ('power_parity_rs_425000_545000',      '₹4.25L–5.45L'),
    ('power_parity_above_rs_545000',       '>₹5.45L'),
]

national_pp = {label: df[col].sum() for col, label in pp_cols}
total_pp = sum(national_pp.values())
pp_pct    = {k: v/total_pp*100 for k, v in national_pp.items()}

fig, ax = plt.subplots(figsize=(12, 4))
palette = sns.color_palette('RdYlGn', len(pp_pct))
left = 0
for (label, pct), color in zip(pp_pct.items(), palette):
    ax.barh('India', pct, left=left, color=color, label=label, edgecolor='white')
    if pct > 3:
        ax.text(left + pct/2, 0, f'{pct:.1f}%', ha='center', va='center', fontsize=8, fontweight='bold')
    left += pct
ax.set_xlabel('% of Households')
ax.set_title('Household Income (Power Parity) Distribution — National Level', fontsize=12, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

## 4.5 — Power Parity Distribution by State (Heatmap %)

In [ ]:
state_pp = state[state['total_power_parity'] > 0].copy()
for col, label in pp_cols:
    state_pp[label] = (state_pp[col] / state_pp['total_power_parity'] * 100).round(1)

heat = state_pp.set_index('state_name')[[label for _, label in pp_cols]]
heat = heat.sort_values('<₹45K', ascending=False)

fig, ax = plt.subplots(figsize=(13, 12))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=.3,
            cbar_kws={'label': '% of Households'}, ax=ax)
ax.set_title('Income (Power Parity) Distribution by State (%) — sorted by lowest income bracket',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 4.6 — Non-Worker Dependency Ratio vs Worker Participation (Scatter)

In [ ]:
scatter_df = df[df['workers'] > 0].copy()

fig = px.scatter(
    scatter_df,
    x='worker_participation', y='dependency_ratio',
    color='state_name', hover_name='district_name',
    labels={'worker_participation':'Worker Participation Rate (%)','dependency_ratio':'Dependency Ratio (Non-workers per 100 workers)'},
    title='Worker Participation Rate vs Dependency Ratio — District Level',
    size='population', size_max=25
)
fig.update_layout(showlegend=False, height=550)
fig.show()